<center>
<img src="https://supportvectors.ai/logo-poster-transparent.png" width=400px style="opacity:0.8">
</center>


In [1]:
%run supportvectors-common.ipynb


<div style="color:#aaa;font-size:8pt">
<hr/>
&copy; SupportVectors. All rights reserved. <blockquote>This notebook is the intellectual property of SupportVectors, and part of its training material. 
Only the participants in SupportVectors workshops are allowed to study the notebooks for educational purposes currently, but is prohibited from copying or using it for any other purposes without written permission.

<b> These notebooks are chapters and sections from Asif Qamar's textbook that he is writing on Data Science. So we request you to not circulate the material to others.</b>
 </blockquote>
 <hr/>
</div>



# Loading and evaluating the fine-tuned relevance detector

In [2]:
from rel_detector.classification.relevance_detector import RelevanceDetector
from rel_detector.classification.relevance_detector import DataProcessor
from rich import print as rprint

In [3]:
model_path = "../../models/relevance_detector"
detector = RelevanceDetector(output_dir=model_path)
detector.load_model(model_path=model_path)
rprint(detector.model)

ModernBertForSequenceClassification(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 1024, padding_idx=50283)
      (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=1024, out_features=3072, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=1024, out_features=1024, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=1024, out_features=5248, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=2624, out_features=1024, bias=False)
        )
      )
      (1-27): 27 x ModernBertEncoderLayer(
        (attn_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=1024, out_features=3072, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=1024, out_features=1024, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=1024, out_features=5248, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=2624, out_features=1024, bias=False)
        )
      )
    )
    (final_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (head): ModernBertPredictionHead(
    (dense): Linear(in_features=1024, out_features=1024, bias=False)
    (act): GELUActivation()
    (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (drop): Dropout(p=0.0, inplace=False)
  (classifier): Linear(in_features=1024, out_features=2, bias=True)
)

In [4]:
data_processor = DataProcessor(data_path="../../data/synthetic_data/synthetic_data.jsonl", split_ratio=0.05)
_, val_dataset = data_processor.load_data()

detector.evaluate_dataset(val_dataset)

Map:   0%|          | 0/265 [00:00<?, ? examples/s]

{'accuracy': 0.9886792452830189,
 'precision': 0.9869281045751634,
 'recall': 0.993421052631579,
 'f1': 0.9901639344262295,
 'threshold': 0.7500000000000002,
 'detailed_report': {'Irrelevant': {'precision': 0.9910714285714286,
   'recall': 0.9823008849557522,
   'f1-score': 0.9866666666666667,
   'support': 113.0},
  'Relevant': {'precision': 0.9869281045751634,
   'recall': 0.993421052631579,
   'f1-score': 0.9901639344262295,
   'support': 152.0},
  'accuracy': 0.9886792452830189,
  'macro avg': {'precision': 0.988999766573296,
   'recall': 0.9878609687936656,
   'f1-score': 0.9884153005464481,
   'support': 265.0},
  'weighted avg': {'precision': 0.9886948804679105,
   'recall': 0.9886792452830189,
   'f1-score': 0.9886726466646047,
   'support': 265.0}}}

In [5]:
# Test predictions
test_texts = [
    "Dolphins have been observed teaching each other complex hunting techniques.",
    "The new smartphone app allows users to track their daily steps.",
    "Elephants have a highly developed brain and are considered one of the smartest animals.",
    "The economy is expected to experience a significant downturn."
]

print("\nTesting predictions:")
for text in test_texts:
    result = detector.predict(text)
    print(f"Text: {text[:50]}...")
    print(f"  Relevant: {result['is_relevant']} (prob: {result['relevance_probability']:.3f})")
    print()


Testing predictions:
Text: Dolphins have been observed teaching each other co...
  Relevant: True (prob: 1.000)

Text: The new smartphone app allows users to track their...
  Relevant: False (prob: 0.000)

Text: Elephants have a highly developed brain and are co...
  Relevant: True (prob: 1.000)

Text: The economy is expected to experience a significan...
  Relevant: False (prob: 0.000)

